## Define NLL Evaluator

In [1]:
import torch
import numpy as np
from tqdm.notebook import tqdm
from transformers import AutoTokenizer, GPTJForCausalLM

class NLLEvaluator:
    def __init__(self, model_id="gpt2", device="cuda"):
        """
        Calculates Negative Log-Likelihood using a pretrained causal language model.
        Using 'gpt2' by default for faster/lighter evaluation. 
        Swap to 'EleutherAI/gpt-j-6b' if you have 12GB+ VRAM.
        """
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        
        print(f"Loading {model_id} for NLL evaluation...")
        self.model = GPTJForCausalLM.from_pretrained(
            model_id,
            revision="float16" if "gpt-j" in model_id else "main",
            dtype=torch.float16,
            low_cpu_mem_usage=True
        ).to(self.device)
        self.model.eval()
        
    def compute(self, samples):
        print("Calculating NLL for each sample...")
        losses = []
        
        with torch.no_grad():
            for sample in tqdm(samples, desc="NLL Evaluation"):
                input_ids = self.tokenizer(sample, return_tensors="pt").input_ids.to(self.device)
                
                if input_ids.shape[1] == 0:
                    continue
                    
                output = self.model(
                    input_ids,
                    return_dict=True,
                    labels=input_ids,
                )
                losses.append(output.loss.item())
                
        return np.mean(losses), losses

## Define Entropy Evaluator

In [2]:
class EntropyEvaluator:
    def __init__(self, tokenizer):
        """
        Calculates Unigram Shannon Entropy based on token frequencies.
        Expects a HuggingFace tokenizer.
        """
        self.tokenizer = tokenizer

    def compute(self, samples):
        print("Calculating Unigram Entropy...")
        sample_stats = {}
        
        for sample in samples:
            tokens = self.tokenizer(sample, return_tensors="pt").input_ids[0]
            for token in tokens:
                token_id = token.item()
                if token_id not in sample_stats:
                    sample_stats[token_id] = 0
                sample_stats[token_id] += 1

        total_tokens = sum(sample_stats.values())
        sample_entropy = 0.0
        
        # Shannon entropy: H(X) = - \sum p(x) \ln p(x)
        for v in sample_stats.values():
            p = v / total_tokens
            if p > 0:
                sample_entropy += -p * np.log(p)
                
        return sample_entropy

## Run Evaluation Pipeline

In [3]:
import os
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"
nll_evaluator = NLLEvaluator(model_id="EleutherAI/gpt-j-6b", device=device)

# We can reuse the tokenizer from the NLL evaluator for the entropy calculations
entropy_evaluator = EntropyEvaluator(tokenizer=nll_evaluator.tokenizer)

df_results = []
samples_dir = "samples"
for temp in [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    samples_path = os.path.join(samples_dir, f"samples_temp_{temp:.1f}.txt")
    if os.path.exists(samples_path):
        with open(samples_path, 'r') as f:
            temp_samples = [line.strip() for line in f.readlines() if line.strip()]
        print(f"\nEvaluating samples from {samples_path}...")
        mean_nll, all_losses = nll_evaluator.compute(temp_samples)
        entropy = entropy_evaluator.compute(temp_samples)
        print(f"Temperature {temp:.1f} - Average NLL: {mean_nll:.4f}, Unigram Entropy: {entropy:.4f}")
        df_results.append({
            "temperature": temp,
            "average_nll": mean_nll,
            "entropy": entropy
        })
    else:
        print(f"Samples file {samples_path} not found. Skipping evaluation for temperature {temp:.1f}.")

# Display results as table
df_results = pd.DataFrame(df_results)
print("\nEvaluation Results:")
display(df_results.style.format({"temperature": "{:.1f}", "average_nll": "{:.4f}", "entropy": "{:.4f}"}))

/home/kogolobo/miniconda3/envs/dl/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/kogolobo/miniconda3/envs/dl/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Loading EleutherAI/gpt-j-6b for NLL evaluation...


Some weights of the model checkpoint at EleutherAI/gpt-j-6b were not used when initializing GPTJForCausalLM: ['transformer.h.0.attn.bias', 'transformer.h.0.attn.masked_bias', 'transformer.h.1.attn.bias', 'transformer.h.1.attn.masked_bias', 'transformer.h.10.attn.bias', 'transformer.h.10.attn.masked_bias', 'transformer.h.11.attn.bias', 'transformer.h.11.attn.masked_bias', 'transformer.h.12.attn.bias', 'transformer.h.12.attn.masked_bias', 'transformer.h.13.attn.bias', 'transformer.h.13.attn.masked_bias', 'transformer.h.14.attn.bias', 'transformer.h.14.attn.masked_bias', 'transformer.h.15.attn.bias', 'transformer.h.15.attn.masked_bias', 'transformer.h.16.attn.bias', 'transformer.h.16.attn.masked_bias', 'transformer.h.17.attn.bias', 'transformer.h.17.attn.masked_bias', 'transformer.h.18.attn.bias', 'transformer.h.18.attn.masked_bias', 'transformer.h.19.attn.bias', 'transformer.h.19.attn.masked_bias', 'transformer.h.2.attn.bias', 'transformer.h.2.attn.masked_bias', 'transformer.h.20.attn.bi


Evaluating samples from samples/samples_temp_0.5.txt...
Calculating NLL for each sample...


NLL Evaluation:   0%|          | 0/512 [00:00<?, ?it/s]

...

NLL Evaluation:   0%|          | 0/512 [00:00<?, ?it/s]

Calculating Unigram Entropy...
Temperature 0.9 - Average NLL: 6.1747, Unigram Entropy: 7.2131

Evaluating samples from samples/samples_temp_1.0.txt...
Calculating NLL for each sample...


NLL Evaluation:   0%|          | 0/512 [00:00<?, ?it/s]

Calculating Unigram Entropy...
Temperature 1.0 - Average NLL: 6.7697, Unigram Entropy: 7.3930

Evaluation Results:


,temperature,average_nll,entropy
0,0.5,3.7662,5.2889
1,0.6,4.3178,5.9250
2,0.7,4.8695,6.4016
3,0.8,5.5376,6.8764
4,0.9,6.1747,7.2131
5,1.0,6.7697,7.3930
